In [ ]:
import altair as alt
import pandas as pd
import numpy as np
from vega_datasets import data
from datetime import datetime

In [ ]:
df = pd.read_csv("all_parameters_daily.csv")

In [ ]:
df.columns[[13, 27]]

In [ ]:
subset = df[['Date Local', 'AQI', 'Latitude', 'Longitude', 'Local Site Name', 'CBSA Name']]
subset.head()

In [ ]:
subset['AQI'].median()

In [ ]:
locations = subset[['Latitude', 'Longitude', 'Local Site Name', 'CBSA Name']].drop_duplicates("Local Site Name").dropna()

In [ ]:
locations

In [ ]:
datetime.today().timetuple().tm_yday

### PARAMS

In [ ]:
# select subsection geographically
brush = alt.selection_interval(name="zoom")

# parameter choices
choice_a = alt.binding_select(options = ['Ozone', 'SO2', 'NO2', 'CO'],
                                     name = "Parameter Name: " )
choice_b = alt.binding_select(options = ['Ozone', 'SO2', 'NO2', 'CO'],
                                     name = "Parameter Name: " )
# time selection

day_slider = alt.binding_range(min=1, max=365, step=1, name="Day: ")
day_select = alt.param(name="selected_day", bind=day_slider, value=5)

# site select
site_selection = alt.selection_point(fields=['Local Site Name'], on = 'mouseover', empty=False)

# selections
selection_a = alt.selection_point(fields = ['Parameter Name'], bind = choice_a)
selection_b = alt.selection_point(fields = ['Parameter Name'], bind = choice_b)

parameters = df[["Date Local", "Parameter Name", "Arithmetic Mean", "AQI", "Local Site Name", "Longitude", "Latitude"]]
parameters["Day"] = [datetime.strptime(i, "%Y-%m-%d").timetuple().tm_yday for i in parameters["Date Local"]]



In [ ]:
parameters.head()

In [ ]:
pollutant_list = sorted(parameters["Parameter Name"].unique().tolist())
pollutant_dropdown = alt.binding_select(
    options=pollutant_list,
    name="Pollutant: ",
)
pollutant_select = alt.param(
    name="pollutant_choice",
    value="Carbon monoxide",
    bind=pollutant_dropdown,
)

### MAIN MAP

In [ ]:
# geomap

usa = data.us_10m.url
geo = alt.layer(

    alt.Chart(alt.topo_feature(usa, 'states')).mark_geoshape(
        fill = '#e6f3f7', stroke = '#000', strokeWidth = 1
        ).project(
            type = 'albersUsa'
        ).properties(
            width = 800, height = 500
        ),

    alt.Chart(locations).mark_circle(size = 8).encode(
        longitude='Longitude:Q',
        latitude='Latitude:Q',
        tooltip=['Local Site Name:N', 'CBSA Name:N'],
        color=alt.condition(site_selection, alt.value('orange'), alt.value('grey'))
    )

).project(
    type = 'albersUsa'
).properties(
    width = 800, height = 500
).add_params(site_selection)


In [ ]:
usa = data.us_10m.url
geo2 = alt.layer(

    alt.Chart(alt.topo_feature(usa, 'states')).mark_geoshape(
        fill = '#e6f3f7', stroke = '#000', strokeWidth = 1
        ).project(
            type = 'albersUsa'
        ).properties(
            width = 800, height = 500
        ),

    alt.Chart(parameters).mark_circle(
        opacity=0.85,
        stroke="#0d1117",
        strokeWidth=0.5,
    ).transform_filter(
        # keep only the selected pollutant
        alt.datum["Parameter Name"] == pollutant_select,
    ).transform_filter(
        # keep only the selected date (exact day match)
        alt.datum["Day"] == day_select,
    ).encode(
        longitude="Longitude:Q",
        latitude="Latitude:Q",
        color=alt.Color(
            "AQI:Q",
            scale=alt.Scale(scheme="magma"),
            legend=alt.Legend(title="Concentration Arithmetic Mean (color)"),
        ),
        tooltip=[
            alt.Tooltip("CBSA Name:N", title="Location"),
            alt.Tooltip("Parameter Name:N", title="Pollutant"),
            alt.Tooltip("Date Local:N", title="Date"),
            alt.Tooltip("Arithmetic Mean:Q", title="Mean", format=".2f"),
        ],
    ).project(
        type="albersUsa"
    )
    
).project(
    type = 'albersUsa'
).properties(
    width = 800, height = 500
).add_params(site_selection, pollutant_select, day_select)


In [ ]:
geo2

In [ ]:
geo2.save("interactive_param_view.html")

### FACET

In [ ]:
# facet
# on hover: take points and compare pollutant levels at that point in time (bar)
# might not have both
# default to us median



In [ ]:
parameters[parameters["Day"]==5]

In [ ]:
parameters.dtypes

In [ ]:
alt.data_transformers.disable_max_rows()
param_bar = alt.Chart(parameters).mark_bar().encode(
    x=alt.X("Parameter Name:N"),
    y=alt.Y("Arithmetic Mean:Q")
).add_params(
    day_select
).transform_filter(
    alt.datum.Day == day_select
).transform_filter(
    site_selection
)

In [ ]:
geo | param_bar

In [ ]:
(geo | param_bar).save("interactive_param_view.html")